In [1]:
!nvidia-smi

Mon Mar  2 00:14:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
import tensorflow as tf
import time

gpus = tf.config.list_physical_devices('GPU')
print(f"Detected GPUs: {len(gpus)}")

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()
x_train = tf.cast(x_train, tf.float32) / 255.0
y_train = tf.cast(y_train, tf.int32)

BATCH_SIZE = 256

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.shuffle(10000).batch(BATCH_SIZE, drop_remainder=True)

def create_cnn_model():
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(100)
    ])

with tf.device('/CPU:0'):
    model = create_cnn_model()
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

train_loss_tracker = tf.keras.metrics.Mean(name='train_loss')
train_acc_tracker = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

@tf.function
def parallel_train_step(x_batch, y_batch):
    mid = tf.shape(x_batch)[0] // 2
    x_gpu0, y_gpu0 = x_batch[:mid], y_batch[:mid]
    x_gpu1, y_gpu1 = x_batch[mid:], y_batch[mid:]

    with tf.device('/GPU:0'):
        with tf.GradientTape() as tape0:
            logits0 = model(x_gpu0, training=True)
            loss0 = loss_fn(y_gpu0, logits0)
        grads0 = tape0.gradient(loss0, model.trainable_variables)

    with tf.device('/GPU:1'):
        with tf.GradientTape() as tape1:
            logits1 = model(x_gpu1, training=True)
            loss1 = loss_fn(y_gpu1, logits1)
        grads1 = tape1.gradient(loss1, model.trainable_variables)

    with tf.device('/CPU:0'):
        averaged_gradients = []
        for g0, g1 in zip(grads0, grads1):
            if g0 is not None and g1 is not None:
                averaged_gradients.append((g0 + g1) / 2.0)
            else:
                averaged_gradients.append(None)
                
        optimizer.apply_gradients(zip(averaged_gradients, model.trainable_variables))

        combined_logits = tf.concat([logits0, logits1], axis=0)
        combined_labels = tf.concat([y_gpu0, y_gpu1], axis=0)
        
        train_loss_tracker.update_state((loss0 + loss1) / 2.0)
        train_acc_tracker.update_state(combined_labels, combined_logits)

EPOCHS = 100

for epoch in range(EPOCHS):
    start_time = time.time()
    
    train_loss_tracker.reset_state()
    train_acc_tracker.reset_state()
    
    for step, (x_batch, y_batch) in enumerate(train_dataset):
        parallel_train_step(x_batch, y_batch)
        
        if step % 50 == 0:
            print(f"  Step {step}: Loss = {train_loss_tracker.result():.4f}, "
                  f"Accuracy = {train_acc_tracker.result():.4f}")

    end_time = time.time()
    print(f"--- Epoch {epoch + 1} Completed (Time: {end_time - start_time:.2f} sec) ---")
    print(f"Epoch {epoch + 1} Result -> Average Loss: {train_loss_tracker.result():.4f}, "
          f"Average Accuracy: {train_acc_tracker.result():.4f}\n")

Detected GPUs: 2
  Step 0: Loss = 4.6232, Accuracy = 0.0156
  Step 50: Loss = 4.4976, Accuracy = 0.0203
  Step 100: Loss = 4.3764, Accuracy = 0.0338
  Step 150: Loss = 4.2379, Accuracy = 0.0531
--- Epoch 1 Completed (Time: 6.87 sec) ---
Epoch 1 Result -> Average Loss: 4.1325, Average Accuracy: 0.0681

  Step 0: Loss = 3.8191, Accuracy = 0.1016
  Step 50: Loss = 3.6040, Accuracy = 0.1457
  Step 100: Loss = 3.5395, Accuracy = 0.1593
  Step 150: Loss = 3.4851, Accuracy = 0.1686
--- Epoch 2 Completed (Time: 4.50 sec) ---
Epoch 2 Result -> Average Loss: 3.4407, Average Accuracy: 0.1762

  Step 0: Loss = 3.2676, Accuracy = 0.2148
  Step 50: Loss = 3.1768, Accuracy = 0.2215
  Step 100: Loss = 3.1544, Accuracy = 0.2262
  Step 150: Loss = 3.1143, Accuracy = 0.2347
--- Epoch 3 Completed (Time: 4.39 sec) ---
Epoch 3 Result -> Average Loss: 3.0836, Average Accuracy: 0.2401

  Step 0: Loss = 2.9837, Accuracy = 0.2109
  Step 50: Loss = 2.9159, Accuracy = 0.2747
  Step 100: Loss = 2.8906, Accuracy = 